In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time


np.random.seed(1)

# Little boilerplate code to find out if we have a gpu
device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")
print(f'Device: {device}')

## Seismic Data Denoising

In [ ]:
## Loading the shot gathers
import numpy as np
data = np.load('data/Marmousi2_Data/Marmousi2_5hz_Data.npy')
data = np.permute_dims(data,(1,0,2))[14:15,:,:]
data.shape

In [ ]:
import torch
from matplotlib.ticker import MaxNLocator

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}
f, ax = plt.subplots(1, 1, figsize=(14, 10), sharey=False)
plt.rc('font', **font)

dx        = 20                  # step interval along x/z direction
dt        = 0.006                 # time interval (e.g., 6ms)
shotnum = 0
selected_shot = torch.tensor(data[shotnum,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))

ax.imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax.set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax.set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax.set_title('Observed Data',fontsize='large', fontweight='bold')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.show()
print('Shot Dimension is:',selected_shot.shape)

## Adding Gaussian Noise

In [ ]:
import torch

# Small value to prevent division by zero in SNR
epsilon = 1e-10

# Example data with shape (30, 1000, 200)
dat = torch.tensor(data)  # Convert to tensor
num_shots = dat.shape[0]
shots_per_noise_level = 10

# Define different levels of noise (standard deviations)
noise_levels = torch.linspace(0.5, 1, shots_per_noise_level)

# Initialize a list to collect shots with noise and SNR
noisy_shots = []
clean_shots = []
snr_values = []

# Add Gaussian noise to each shot and calculate SNR
for i in range(num_shots):
    for noise_level in noise_levels:
        noise = torch.normal(mean=0.0, std=noise_level, size=dat[i, :, :].shape)
        clean_shot = dat[i, :, :]
        noisy_shot = dat[i, :, :] + noise
        
        # Calculate SNR
        signal_power = torch.mean(dat[i, :, :]**2)  # Power of the signal
        noise_power = torch.mean(noise**2) + epsilon  # Power of the noise with epsilon
        snr = 10 * torch.log10(signal_power / noise_power)  # SNR in dB   
        snr_values.append(snr.item())
        noisy_shots.append(noisy_shot)
        clean_shots.append(clean_shot)

# Convert list to a PyTorch tensor
data_noisy = torch.stack(noisy_shots)
data_clean = torch.stack(clean_shots)
data_noisy = data_noisy.reshape((data_noisy.shape[0],1,data_noisy.shape[1],data_noisy.shape[2]))
data_clean = data_clean.reshape((data_clean.shape[0],1,data_clean.shape[1],data_clean.shape[2]))

# Check the shape and SNR values
print('The number of noisy data:', data_noisy.shape)  # Shape of the noisy data
print(snr_values[0:10])  # Print the SNR for each noisy shot


## Plotting Noisy Data

In [ ]:
import torch
from matplotlib.ticker import MaxNLocator

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}
f, ax = plt.subplots(1, 1, figsize=(14, 10), sharey=False)
plt.rc('font', **font)

dx        = 20                  # step interval along x/z direction
dt        = 0.006                 # time interval (e.g., 6ms)
shotnum = 5
selected_shot = torch.tensor(data_noisy[shotnum,0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))

ax.imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax.set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax.set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax.set_title('Observed Data',fontsize='large', fontweight='bold')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.show()
print('Shot Dimension is:',selected_shot.shape)

## UNET

In [ ]:
import torch
import torch.nn as nn

class UNET(nn.Module):
    def __init__(self, input_channels, FM1, FM2, FM3, bottleneck_dim, output_channels):
        super(UNET, self).__init__()
        
        # Encoder
        self.encoder_conv1 = nn.Conv2d(input_channels, FM1, kernel_size=3, stride=1, padding=1)
        self.encoder_ac1 = nn.LeakyReLU(0.1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder_conv2 = nn.Conv2d(FM1, FM2, kernel_size=3, stride=1, padding=1)  # Same feature maps
        self.encoder_ac2 = nn.LeakyReLU(0.1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder_conv3 = nn.Conv2d(FM2, FM3, kernel_size=3, stride=1, padding=1)
        self.encoder_ac3 = nn.LeakyReLU(0.1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder_bottleneck = nn.Conv2d(FM3, bottleneck_dim, kernel_size=3, stride=1, padding=1)  # Bottleneck with 1 feature map
        self.encoder_bottleneck_ac = nn.LeakyReLU(0.1)

        # Decoder
        self.decoder_upsample1 = nn.Upsample(scale_factor=2, mode='bilinear')
        self.decoder_conv1 = nn.Conv2d(bottleneck_dim, FM3, kernel_size=3, padding=1)
        self.decoder_ac1 = nn.LeakyReLU(0.1)

        self.decoder_upsample2 = nn.Upsample(scale_factor=2, mode='bilinear')
        self.decoder_conv2 = nn.Conv2d(FM3, FM2, kernel_size=3, padding=1)
        self.decoder_ac2 = nn.LeakyReLU(0.1)

        self.decoder_upsample3 = nn.Upsample(scale_factor=2, mode='bilinear')
        self.decoder_conv3 = nn.Conv2d(FM2, FM1, kernel_size=3, padding=1)
        self.decoder_ac3 = nn.LeakyReLU(0.1)

        self.decoder_convF = nn.Conv2d(FM1, output_channels, kernel_size=3, padding=1)  # Final output layer
        #self.decoder_acF = nn.Tanh()

    def forward(self, x):
        # Encoding
        E1 = self.encoder_conv1(x)
        E1 = self.encoder_ac1(E1)
        P1 = self.pool1(E1)

        E2 = self.encoder_conv2(P1)
        E2 = self.encoder_ac2(E2)
        P2 = self.pool2(E2)

        E3 = self.encoder_conv3(P2)
        E3 = self.encoder_ac3(E3)
        P3 = self.pool3(E3)

        # Bottleneck
        B = self.encoder_bottleneck(P3)
        B = self.encoder_bottleneck_ac(B)

        # Decoding
        D1 = self.decoder_upsample1(B)
        D1 = self.decoder_conv1(D1)
        D1 = self.decoder_ac1(D1)
        #D1 = D1 + E3  # Skip connection

        D2 = self.decoder_upsample2(D1)
        D2 = self.decoder_conv2(D2)
        D2 = self.decoder_ac2(D2)
        #D2 = D2 + E2  # Skip connection

        D3 = self.decoder_upsample3(D2)
        D3 = self.decoder_conv3(D3)
        D3 = self.decoder_ac3(D3)
        #D3 = D3 + E1  # Skip connection
        
        out = self.decoder_convF(D3)  # Final output layer
        #out = self.decoder_acF(out)
        
        return B, out

## Initialization of network parameters


In [ ]:
import torch.nn.init as init

def weights_init(m, leak_value):
    
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        if m.weight is not None:

            init.kaiming_normal_(m.weight, a = leak_value)

        if m.bias is not None:
            init.constant_(m.bias, 0.0)

    if isinstance(m, nn.Linear):
        if m.weight is not None:
            init.kaiming_normal_(m.weight, a = leak_value)
        if m.bias is not None:
            init.constant_(m.bias, 0.0)

## Data preparation

In [ ]:
from torch.utils.data import Dataset, DataLoader

# 1. Define your Dataset class (if you haven't already):

class MyDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)  # Convert to torch tensor
        self.labels = torch.tensor(labels, dtype=torch.float32) #Convert labels to long for classification

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
import torch
from matplotlib.ticker import MaxNLocator
def plotting(net_input,outputs,data_noisy1):

    font = {'family' : 'normal',
            'weight' : 'bold',
            'size'   : 10}
    f, ax = plt.subplots(1, 4, figsize=(14, 10), sharey=False)
    plt.rc('font', **font)

    dx        = 20                  # step interval along x/z direction
    dt        = 0.006                 # time interval (e.g., 6ms)
    shotnum = 0

    # Noisy data (input)
    selected_shot = torch.tensor(net_input[shotnum,:,:][0,:,:]).to(device)
    vmin, vmax = torch.quantile(selected_shot,
                                torch.tensor([0.05, 0.95]).to(device))

    ax[0].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
                 vmin=vmin, vmax=vmax,
                 extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

    ax[0].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
    ax[0].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
    ax[0].set_title('Input Noise',fontsize='large', fontweight='bold')
    ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

    # Denoised data
    selected_shot = torch.tensor(outputs[shotnum,:,:][0,:,:]).to(device)
    vmin, vmax = torch.quantile(selected_shot,
                                torch.tensor([0.05, 0.95]).to(device))
    ax[1].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
                 vmin=vmin, vmax=vmax,
                 extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

    ax[1].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
    ax[1].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
    ax[1].set_title('Denoised Data',fontsize='large', fontweight='bold')
    ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))


    # Target data
    selected_shot = torch.tensor(data_noisy1[shotnum,:,:][0,:,:]).to(device)
    vmin, vmax = torch.quantile(selected_shot,
                                torch.tensor([0.05, 0.95]).to(device))
    ax[2].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
                 vmin=vmin, vmax=vmax,
                 extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

    ax[2].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
    ax[2].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
    ax[2].set_title('Target Data',fontsize='large', fontweight='bold')
    ax[2].xaxis.set_major_locator(MaxNLocator(integer=True))

    # Error
    selected_shot = torch.tensor(outputs[shotnum,:,:][0,:,:]-data_noisy1[shotnum,:,:][0,:,:]).to(device)
    vmin, vmax = torch.quantile(selected_shot,
                                torch.tensor([0.05, 0.95]).to(device))
    ax[3].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
                 vmin=vmin, vmax=vmax,
                 extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

    ax[3].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
    ax[3].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
    ax[3].set_title('Error',fontsize='large', fontweight='bold')
    ax[3].xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

data_noisy1 = data_noisy[5:6,:,:].to(device)
data_clean1 = data_clean[5:6,:,:].to(device)
shape = [data_noisy1.shape[0],data_noisy1.shape[1],data_noisy1.shape[2],data_noisy1.shape[3]]
net_input = torch.zeros(shape)
reg_noise_std = 1
net_input = net_input.normal_().to(device)
net_input *= reg_noise_std
#net_input *= data_noisy1.abs().max() 
batch_size = 1


In [ ]:

input_channels = 1
FM1 = 64
FM2 = 2*FM1
FM3 = 2*FM2
bottleneck_dim = 2*FM3
output_channels = 1
net_input_saved = net_input.detach().clone()
noise = net_input.detach().clone()

model = UNET(input_channels, FM1, FM2, FM3, bottleneck_dim, output_channels)
### init the hyper-parameters of model ###
#leak_value = 10
#model.apply(lambda m: weights_init(m, leak_value))
model = model.to(device)
print(model)


criterion = nn.MSELoss()
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  

# Train the model
model.train()
num_epochs = 6000

for epoch in range(num_epochs):
    # Load images with gradient accumulation capabilities
    net_input = net_input_saved + (noise.normal_() * reg_noise_std)
    # Clear gradients w.r.t. parameters
    optimizer.zero_grad()

    # Forward pass to get output/logits
    E, outputs = model(net_input)

    # Calculate Loss
    loss = criterion(outputs[0], data_noisy1)
    losall = loss.item()

    # Getting gradients w.r.t. parameters
    loss.backward()

    # Updating parameters
    optimizer.step()
    
    if epoch%500==0:
        plotting(net_input,outputs,data_noisy1)



    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
#torch.save(model.state_dict(), 'data/Marmousi2_Data/model_weights_DIP.pth')


In [ ]:
# Load Best Model
#model.load_state_dict(torch.load('data/Marmousi2_Data/model_weights_DIP.pth'))
model.eval()
laball = []
dat = []
outputsall= []
Eall = []
# Iterate through test dataset
# Load images with gradient accumulation capabilities


# Forward pass only to get logits/output
E, outputs = model(net_input)
loss = criterion(outputs[0], data_noisy1)

# Print Loss
print('Loss: {}'.format(loss.item()))

## Plotting the denoised results of the test set

In [ ]:
import torch
from matplotlib.ticker import MaxNLocator

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}
f, ax = plt.subplots(1, 4, figsize=(14, 10), sharey=False)
plt.rc('font', **font)

dx        = 20                  # step interval along x/z direction
dt        = 0.006                 # time interval (e.g., 6ms)
shotnum = 0

# Noisy data (input)
selected_shot = torch.tensor(net_input[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))

ax[0].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[0].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[0].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[0].set_title('Input Noise',fontsize='large', fontweight='bold')
ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

# Denoised data
selected_shot = torch.tensor(outputs[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[1].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[1].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[1].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[1].set_title('Denoised Data',fontsize='large', fontweight='bold')
ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))


# Target data
selected_shot = torch.tensor(data_noisy1[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[2].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[2].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[2].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[2].set_title('Target Data',fontsize='large', fontweight='bold')
ax[2].xaxis.set_major_locator(MaxNLocator(integer=True))

# Error
selected_shot = torch.tensor(outputs[shotnum,:,:][0,:,:]-data_noisy1[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[3].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[3].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[3].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[3].set_title('Error',fontsize='large', fontweight='bold')
ax[3].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()


# Calculate SNR befor denoisning


In [ ]:
difference_power = torch.mean((data_noisy1[shotnum,:,:][0,:,:]-data_clean1[shotnum,:,:][0,:,:])**2)  # Power of the signal
noisy_power = torch.mean(data_noisy1[shotnum,:,:][0,:,:]**2) + epsilon  # Power of the noise with epsilon
snr = 10 * torch.log10(noisy_power / difference_power)  # SNR in dB 
print('SNR before denoising:',snr)

# Calculate SNR after denoisning


In [ ]:
difference_power = torch.mean((outputs[shotnum,:,:][0,:,:]-data_clean1[shotnum,:,:][0,:,:])**2)  # Power of the signal
noisy_power = torch.mean(outputs[shotnum,:,:][0,:,:]**2) + epsilon  # Power of the noise with epsilon
snr = 10 * torch.log10(noisy_power / difference_power)  # SNR in dB 
print('SNR after denoising:',snr)


## Modified DIP (Advanced Self-Supervised)

## PatchUNET Class

In [ ]:
import torch 
import torch.nn as nn

class PatchUNET(nn.Module):
    def __init__(self):
        super(PatchUNET, self).__init__()


               
        self.e1 = nn.Linear(48*48, 128)
        self.ae1 = nn.ELU(1) 

        self.e2 = nn.Linear(128, 64)
        self.ae2 = nn.ELU(1) 
        
        self.e3 = nn.Linear(64, 32)
        self.ae3 = nn.ELU(1) 
        
        self.e4 = nn.Linear(32, 16)
        self.ae4 = nn.ELU(1) 
        
        self.e5 = nn.Linear(16, 8)
        self.ae5 = nn.ELU(1) 
        
        self.e6 = nn.Linear(8, 4)
        self.ae6 = nn.ELU(1) 

        
        self.d1 = nn.Linear(4, 4)
        self.de1 = nn.ELU(1) 

        self.d2 = nn.Linear(8, 8)
        self.de2 = nn.ELU(1) 
        
        self.d3 = nn.Linear(16, 16)
        self.de3 = nn.ELU(1) 
        
        self.d4 = nn.Linear(32, 32)
        self.de4 = nn.ELU(1) 
        
        self.d5 = nn.Linear(64, 64)
        self.de5 = nn.ELU(1) 
        
        self.d6 = nn.Linear(128, 128)
        self.de6 = nn.ELU(1) 
        
        self.sec = nn.Linear(256,48*48)
        
    def forward(self, inputs):

        e1 = self.ae1(self.e1(inputs))
        e2 = self.ae2(self.e2(e1))
        e3 = self.ae3(self.e3(e2))
        e4 = self.ae4(self.e4(e3))
        e5 = self.ae5(self.e5(e4))
        e6 = self.ae6(self.e6(e5))
        
        
        d1 = self.de1(self.d1(e6))
        d1 = torch.cat((d1,e6),axis=-1)
        
        #print(d1.size())
        d2 = self.de2(self.d2(d1))
        d2 = torch.cat((d2,e5),axis=-1)
        
        #print(d2.size())
        d3 = self.de3(self.d3(d2))
        d3 = torch.cat((d3,e4),axis=-1)
        
        #print(d3.size())
        d4 = self.de4(self.d4(d3))
        d4 = torch.cat((d4,e3),axis=-1)
        
        #print(d4.size())
        d5 = self.de5(self.d5(d4))
        d5 = torch.cat((d5,e2),axis=-1)
        
        #print(d5.size())        
        d6 = self.de6(self.d6(d5))
        d6 = torch.cat((d6,e1),axis=-1)
        
        #print(d6.size())
        out = self.sec(d6)
        
        
        return out 
    
netD = PatchUNET()
netD = netD.to(device)



## Patching

In [ ]:
import torch.nn.functional as F

def patchify(inp, kernel_size, stride, inv=False, orig_size=None):
    if not inv:
        out = inp.unfold(0, kernel_size[0], stride[0]).unfold(1, kernel_size[1], stride[1])
        out = out.reshape(-1, kernel_size[0], kernel_size[1])
        input_ones = torch.ones_like(inp).unfold(0, kernel_size[0], stride[0]).unfold(1, kernel_size[1], stride[1])
        divisor = input_ones.reshape(-1, kernel_size[0], kernel_size[1])

        return out

    elif inv and orig_size is not None:
        out = inp.reshape(-1, kernel_size[0]*kernel_size[1])
        out = F.fold(out.transpose(0, 1), output_size=orig_size, kernel_size=kernel_size, stride=stride)[0]
        divisor = torch.ones_like(inp)
        divisor = divisor.reshape(-1, kernel_size[0]*kernel_size[1])
        divisor = F.fold(divisor.transpose(0, 1), output_size=orig_size, kernel_size=kernel_size, stride=stride)[0]

        return out / divisor
    

w1 = 48
w2 = 48
s1z = 8
s2z = 8
  
# Patching the noisy data.

dataInput = data_noisy1[0,0,:,:].cpu()
# If you want to normalize
#ma = dataInput.abs().max()
#dataInput = dataInput/ma

dataInputP = patchify(torch.tensor(dataInput),(w1,w2),(s1z,s2z)).reshape(-1, w1*w2).numpy() 
dataInput2 = np.reshape(dataInputP,(dataInputP.shape[0],1,w1,w2))


dataInput2 = torch.tensor(dataInput2).float()

print('data size:', dataInput2.shape)


In [ ]:

batch_size = 128
# data and labels to tensor
datainput = torch.tensor(dataInput2,dtype=torch.float32).to(device)


# Create your Dataset instance
dataset_train = MyDataset(datainput, datainput)

# Create a DataLoader
# Adjust batch_size as needed; shuffle=True for training
train_loader = torch.utils.data.DataLoader(dataset=dataset_train, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=dataset_train, 
                                           batch_size=batch_size, 
                                           shuffle=False)

In [ ]:
from tqdm import tqdm

# Train the model
netD.train()
num_epochs = 100
optim_d = torch.optim.Adam(netD.parameters(),lr=1e-3)

for epoch in tqdm(range(num_epochs)):

    losall = 0
    for i, (images, labels) in enumerate(train_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], 1, images.shape[2]*images.shape[3]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0], 1, labels.shape[2]*labels.shape[3]).to(device)

        # Clear gradients w.r.t. parameters
        optim_d.zero_grad()  
        
        # Forward pass to get output/logits
        outputs = netD(images)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        losall+= loss.item()
        
        # Getting gradients w.r.t. parameters
        loss.backward()
        
        # Updating parameters
        optim_d.step()
        
    
            
    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
#torch.save(netD, './data/Marmousi2_Data/model_weights_Denoise_PatchUNET.pt')


In [ ]:

netD.eval()
#netD = torch.load('./data/Marmousi2_Data/model_weights_Denoise_PatchUNET.pt')
preall = []

for epoch in tqdm(range(1)):
    lossall = []
    for i, (images, labels) in enumerate(test_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], 1, images.shape[2]*images.shape[3]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0], 1, labels.shape[2]*labels.shape[3]).to(device)
        
        
        pred_x0 = netD(images)
        # Convert pred_x0 to numpy and append to preall
        preall.append(pred_x0.detach().cpu().numpy())
        

# After the loop, concatenate all the predictions along axis 0
preall = np.concatenate(preall, axis=0)
print(preall.shape)



## Unpatching

In [ ]:
out = np.reshape(preall,(preall.shape[0],w1*w2))     
n1,n2=np.shape(data_noisy1[0,0,:,:])
outB = patchify(torch.tensor(out), (w1,w2), (s1z,s2z), inv=True, orig_size=(n1,n2))
outB = np.array(outB)


outB.shape

In [ ]:
import torch
from matplotlib.ticker import MaxNLocator

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}
f, ax = plt.subplots(1, 4, figsize=(14, 10), sharey=False)
plt.rc('font', **font)

dx        = 20                  # step interval along x/z direction
dt        = 0.006                 # time interval (e.g., 6ms)
shotnum = 0

# Noisy data (input)
selected_shot = torch.tensor(data_clean1[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))

ax[0].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[0].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[0].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[0].set_title('Clean',fontsize='large', fontweight='bold')
ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

# Denoised data
selected_shot = torch.tensor(outB).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[1].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[1].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[1].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[1].set_title('Denoised Data',fontsize='large', fontweight='bold')
ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))


# Target data
selected_shot = torch.tensor(data_noisy1[shotnum,:,:][0,:,:]).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[2].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[2].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[2].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[2].set_title('Target Data',fontsize='large', fontweight='bold')
ax[2].xaxis.set_major_locator(MaxNLocator(integer=True))

# Error
selected_shot = torch.tensor(outB-data_noisy1[shotnum,:,:][0,:,:].cpu().numpy()).to(device)
vmin, vmax = torch.quantile(selected_shot,
                            torch.tensor([0.05, 0.95]).to(device))
ax[3].imshow(selected_shot.cpu(), aspect='auto', cmap='gray',
             vmin=vmin, vmax=vmax,
             extent=[0, selected_shot.shape[1] * dx/1000, selected_shot.shape[0] *(dt), 0])

ax[3].set_xlabel('Position (km)',fontsize='large', fontweight='bold')
ax[3].set_ylabel('Time (s)',fontsize='large', fontweight='bold')
ax[3].set_title('Error',fontsize='large', fontweight='bold')
ax[3].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()


In [ ]:
difference_power = torch.mean((data_noisy1[shotnum,:,:][0,:,:]-data_clean1[shotnum,:,:][0,:,:])**2)  # Power of the signal
noisy_power = torch.mean(data_noisy1[shotnum,:,:][0,:,:]**2) + epsilon  # Power of the noise with epsilon
snr = 10 * torch.log10(noisy_power / difference_power)  # SNR in dB 
print('SNR before denoising:',snr)

In [ ]:
difference_power = torch.mean((torch.tensor(outB).to(device)-data_clean1[shotnum,:,:][0,:,:])**2)  # Power of the signal
noisy_power = torch.mean(torch.tensor(outB)**2) + epsilon  # Power of the noise with epsilon
snr = 10 * torch.log10(noisy_power / difference_power)  # SNR in dB 
print('SNR after denoising:',snr)

## PatchUNET Ref: https://onlinelibrary.wiley.com/doi/full/10.1111/1365-2478.13062